In [ ]:
import scanpy as sc
import os
import matplotlib.pyplot as plt

CLUSTER_KEY = 'joint_alpha_0.9'
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/joint_graph_v2")

# ── Load the source files that have correct spatial metadata ─────────
# Erickson — has uns['spatial'] from SpaceRanger via sc.read_visium
adata_erickson_raw = sc.read_h5ad(
    "/path/to/data/"
    "Analyses_Erickson/Erickson_2022_processed_files/"
    "erickson_2022_raw_qc.h5ad",
    backed='r'  # read metadata only — don't load full matrix into RAM
)
print("Erickson spatial keys:", list(adata_erickson_raw.uns.get('spatial', {}).keys()))

# ── Check what spatial keys exist ────────────────────────────────────
for key in adata_erickson_raw.uns.get('spatial', {}).keys():
    sf = adata_erickson_raw.uns['spatial'][key].get('scalefactors', {})
    print(f"  {key}: scalefactors = {sf}")

In [ ]:
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)
print("Loaded:", adata.shape)
print("Cluster columns:", [c for c in adata.obs.columns if 'hc_ward' in c])

In [ ]:
import json
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pandas as pd
import scanpy as sc

CLUSTER_KEY = 'joint_alpha_0.9'
JUNYI_DIR = ("/path/to/data/"
             "Junyi_2025/visium/")
IMG_ERICKSON_BASE = ("/path/to/data/"
                     "Erickson_2022_Data/svw96g68dv-1/Histological_images/")
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/niche_biology")
os.makedirs(OUT_DIR, exist_ok=True)
Image.MAX_IMAGE_PIXELS = None

# ── Load main adata ──────────────────────────────────────────────────
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

# ── Lock colors globally ─────────────────────────────────────────────
adata.obs[CLUSTER_KEY] = pd.Categorical(
    adata.obs[CLUSTER_KEY],
    categories=sorted(adata.obs[CLUSTER_KEY].unique(),
                      key=lambda x: int(x))
)
cmap_global = cm.get_cmap('tab10', 7)
global_colors = [mcolors.to_hex(cmap_global(i)) for i in range(7)]
adata.uns[f'{CLUSTER_KEY}_colors'] = global_colors

# Build colour map for matplotlib plots
niches = sorted(adata.obs[CLUSTER_KEY].unique(), key=lambda x: int(x))
colour_map = {n: mcolors.to_rgba(global_colors[int(n)]) for n in niches}

# ── Erickson — matplotlib approach ──────────────────────────────────
def get_erickson_img_path(slide_id):
    parts = slide_id.split('_')
    patient = f"{parts[0]}_{parts[1]}"
    slide_name = '_'.join(parts[2:])
    
    if 'V' in slide_id:
        # Check both possible Visium paths
        path1 = os.path.join(IMG_ERICKSON_BASE, patient, 'Visium',
                             f'{slide_name}.jpg')
        path2 = os.path.join(IMG_ERICKSON_BASE, patient,
                             f'{slide_name}.jpg')
        if os.path.exists(path1):
            return path1
        elif os.path.exists(path2):
            return path2
        else:
            print(f"  WARNING: image not found at {path1} or {path2}")
            return path1
    else:
        return os.path.join(IMG_ERICKSON_BASE, patient,
                            f'{slide_name}.jpg')

def plot_erickson_slide(slide_id):
    img_path = get_erickson_img_path(slide_id)
    print(f"  Loading: {img_path}")

    img = Image.open(img_path)
    original_width, original_height = img.size
    print(f"  Original image size: {original_width} x {original_height}")

    # Use larger thumbnail size for big images
    # Thumbnail size must be >= max coord value after scaling
    # For a 24000px image with coords up to ~42000px in full res
    # we need thumbnail large enough to contain all spots
    max_coord_x = adata[adata.obs['slide_id'] == slide_id].obsm['spatial'][:, 0].max()
    max_coord_y = adata[adata.obs['slide_id'] == slide_id].obsm['spatial'][:, 1].max()
    print(f"  Max coords (full res): X={max_coord_x:.0f}, Y={max_coord_y:.0f}")

    # Set thumbnail size to be proportional — keep image covering all spots
    thumb_size = int(original_width)  # no downsampling — use full size
    # But that's too big — instead scale coords to image, not image to coords
    # Just downsample image and scale coords by the SAME factor
    thumb_max = 8000  # increase from 4000 to 8000 for large images
    img.thumbnail((thumb_max, thumb_max), Image.LANCZOS)
    img_arr = np.array(img)

    scale = img_arr.shape[1] / original_width
    print(f"  Scale factor: {scale:.6f}")
    print(f"  Downsampled to: {img_arr.shape}")

    mask = adata.obs['slide_id'] == slide_id
    adata_slide = adata[mask]
    coords = adata_slide.obsm['spatial'].copy() * scale

    print(f"  Coord X range after scaling: {coords[:,0].min():.0f} to {coords[:,0].max():.0f}")
    print(f"  Coord Y range after scaling: {coords[:,1].min():.0f} to {coords[:,1].max():.0f}")
    print(f"  Image dimensions: {img_arr.shape[1]} x {img_arr.shape[0]}")

    labels = adata_slide.obs[CLUSTER_KEY].values
    colours = [colour_map[l] for l in labels]

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    axes[0].imshow(img_arr)
    axes[0].set_title(f'{slide_id}\nH&E only', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(img_arr)
    axes[1].scatter(coords[:, 0], coords[:, 1],
                    c=colours, s=8, alpha=0.8,
                    rasterized=True, linewidths=0)

    handles = [plt.scatter([], [], c=[colour_map[n]], s=30,
                           label=f'Niche {n}') for n in niches]
    axes[1].legend(handles=handles, bbox_to_anchor=(1.01, 1),
                   loc='upper left', fontsize=9, title='Joint graph α=0.9')
    axes[1].set_title(f'{slide_id}\nJoint graph α=0.9', fontsize=12)
    axes[1].axis('off')

    plt.suptitle(f'Erickson 2022 — Spatial niche map — {slide_id}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f'HE_overlay_{slide_id}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out_path}")

# ── Hu — sc.pl.spatial approach ──────────────────────────────────────
def load_hu_spatial_metadata(slide_id):
    spatial_dir = os.path.join(JUNYI_DIR, slide_id, 'spatial')
    with open(os.path.join(spatial_dir, 'scalefactors_json.json')) as f:
        scalefactors = json.load(f)
    img = np.array(Image.open(
        os.path.join(spatial_dir, 'tissue_hires_image.png')))
    print(f"  {slide_id} scalef={scalefactors['tissue_hires_scalef']:.6f} "
          f"image={img.shape}")
    return {'images': {'hires': img}, 'scalefactors': scalefactors}

def plot_hu_slide(slide_id):
    spatial_metadata = load_hu_spatial_metadata(slide_id)
    mask = adata.obs['slide_id'] == slide_id
    adata_slide = adata[mask].copy()
    adata_slide.uns['spatial'] = {slide_id: spatial_metadata}
    adata_slide.uns[f'{CLUSTER_KEY}_colors'] = global_colors

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    sc.pl.spatial(adata_slide, library_id=slide_id,
                  color=None, ax=axes[0], show=False,
                  title=f'{slide_id}\nH&E only')
    sc.pl.spatial(adata_slide, library_id=slide_id,
                  color=CLUSTER_KEY, ax=axes[1], show=False,
                  title=f'{slide_id}\nJoint graph α=0.9')

    plt.suptitle(f'Hu 2025 — Spatial niche map — {slide_id}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f'HE_overlay_{slide_id}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out_path}")

# ── Run all slides ────────────────────────────────────────────────────
erickson_slides = ['Patient_1_H1_2', 'Patient_1_H1_4','Patient_1_H2_2']
hu_slides = ['HP09_CZ_ST', 'HP05_PZ_ST']

print("Plotting Erickson slides...")
for slide_id in erickson_slides:
    print(f"\n{slide_id}...")
    plot_erickson_slide(slide_id)

print("\nPlotting Hu slides...")
for slide_id in hu_slides:
    print(f"\n{slide_id}...")
    plot_hu_slide(slide_id)

print("\nAll done.")

Print Erickson malignant slide H1_5:

In [ ]:
plot_erickson_slide('Patient_1_H1_5')

New code to plot the niche names and have same colours for both (not sure if it works, 15/05):



In [ ]:
import json
import numpy as np
import pandas as pd
from PIL import Image
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import scanpy as sc

# --- Configuration ---
CLUSTER_KEY = 'joint_alpha_0.9'
JUNYI_DIR = "/path/to/data/Junyi_2025/visium/"
IMG_ERICKSON_BASE = "/path/to/data/Erickson_2022_Data/svw96g68dv-1/Histological_images/"
OUT_DIR = "/path/to/data/Cell2location/spatial_mapping/exploration_mean/niche_biology"
os.makedirs(OUT_DIR, exist_ok=True)
Image.MAX_IMAGE_PIXELS = None

# 1. Define the Niche Mapping
NICHE_NAMES = {
    '0': 'Niche 0\nLow-quality/acellular',
    '1': 'Niche 1\nPeriurethral ductal',
    '2': 'Niche 2\nAggressive tumour',
    '3': 'Niche 3\nNecrotic/Low-quality',
    '4': 'Niche 4\nFibromuscular stroma',
    '5': 'Niche 5\nIncidental cancer',
    '6': 'Niche 6\nLuminal epithelium'
}

# --- Load main adata ---
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

# 2. Update Column with Names and Set Categorical Order
# Convert to string first to ensure mapping works, then map, then set as categorical
adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str).map(NICHE_NAMES)

# Define the order based on the original IDs 0-6
ordered_names = [NICHE_NAMES[str(i)] for i in range(7)]
adata.obs[CLUSTER_KEY] = pd.Categorical(
    adata.obs[CLUSTER_KEY], 
    categories=ordered_names, 
    ordered=True
)

# 3. Lock colors globally
# We use tab10 and assign the first 7 colors to our ordered names
cmap_global = cm.get_cmap('tab10', 10)
global_colors = [mcolors.to_hex(cmap_global(i)) for i in range(len(ordered_names))]
adata.uns[f'{CLUSTER_KEY}_colors'] = global_colors

# Build a color map dictionary for the manual Matplotlib plots
colour_map = {name: global_colors[i] for i, name in enumerate(ordered_names)}

# --- Erickson — matplotlib approach ---
def get_erickson_img_path(slide_id):
    parts = slide_id.split('_')
    patient = f"{parts[0]}_{parts[1]}"
    slide_name = '_'.join(parts[2:])
    
    if 'V' in slide_id:
        path1 = os.path.join(IMG_ERICKSON_BASE, patient, 'Visium', f'{slide_name}.jpg')
        path2 = os.path.join(IMG_ERICKSON_BASE, patient, f'{slide_name}.jpg')
        return path1 if os.path.exists(path1) else path2
    else:
        return os.path.join(IMG_ERICKSON_BASE, patient, f'{slide_name}.jpg')

def plot_erickson_slide(slide_id):
    img_path = get_erickson_img_path(slide_id)
    print(f"  Loading: {img_path}")

    img = Image.open(img_path)
    original_width, _ = img.size
    
    # Scale image for plotting
    thumb_max = 8000 
    img.thumbnail((thumb_max, thumb_max), Image.LANCZOS)
    img_arr = np.array(img)
    scale = img_arr.shape[1] / original_width

    mask = adata.obs['slide_id'] == slide_id
    adata_slide = adata[mask]
    coords = adata_slide.obsm['spatial'].copy() * scale

    labels = adata_slide.obs[CLUSTER_KEY].values
    # Get colors based on the renamed labels
    plot_colours = [colour_map[l] for l in labels]

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    axes[0].imshow(img_arr)
    axes[0].set_title(f'{slide_id}\nH&E only', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(img_arr)
    axes[1].scatter(coords[:, 0], coords[:, 1],
                    c=plot_colours, s=8, alpha=0.8,
                    rasterized=True, linewidths=0)

    # Use the categorical names for the legend
    handles = [plt.scatter([], [], c=[colour_map[name]], s=30,
                           label=name) for name in ordered_names]
    
    axes[1].legend(handles=handles, bbox_to_anchor=(1.01, 1),
                   loc='upper left', fontsize=9, title='Niches (Joint graph α=0.9)')
    axes[1].set_title(f'{slide_id}\nJoint graph α=0.9', fontsize=12)
    axes[1].axis('off')

    plt.suptitle(f'Erickson 2022 — Spatial niche map — {slide_id}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f'HE_overlay_{slide_id}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out_path}")

# --- Hu — sc.pl.spatial approach ---
def load_hu_spatial_metadata(slide_id):
    spatial_dir = os.path.join(JUNYI_DIR, slide_id, 'spatial')
    with open(os.path.join(spatial_dir, 'scalefactors_json.json')) as f:
        scalefactors = json.load(f)
    img = np.array(Image.open(os.path.join(spatial_dir, 'tissue_hires_image.png')))
    return {'images': {'hires': img}, 'scalefactors': scalefactors}

def plot_hu_slide(slide_id):
    spatial_metadata = load_hu_spatial_metadata(slide_id)
    mask = adata.obs['slide_id'] == slide_id
    adata_slide = adata[mask].copy()
    adata_slide.uns['spatial'] = {slide_id: spatial_metadata}

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    sc.pl.spatial(adata_slide, library_id=slide_id,
                  color=None, ax=axes[0], show=False,
                  title=f'{slide_id}\nH&E only')
    # sc.pl.spatial will now use the names and global_colors automatically
    sc.pl.spatial(adata_slide, library_id=slide_id,
                  color=CLUSTER_KEY, ax=axes[1], show=False,
                  title=f'{slide_id}\nJoint graph α=0.9')

    plt.suptitle(f'Hu 2025 — Spatial niche map — {slide_id}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f'HE_overlay_{slide_id}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out_path}")

# --- Execution ---
erickson_slides = ['Patient_1_H1_2', 'Patient_1_H1_4','Patient_1_H2_2', 'Patient_1_H1_5', 'Patient_1_H2_1','Patient_1_V1_2']
hu_slides = ['HP09_CZ_ST', 'HP05_PZ_ST','HP02_TZ_ST','HP12_ST', 'HP14_ST','HP02_PZ_ST']

for slide in erickson_slides:
    plot_erickson_slide(slide)

for slide in hu_slides:
    plot_hu_slide(slide)